# Mini-Project: Multi-Platform Content Team

Give the team a topic. They produce posts for LinkedIn, Instagram, and YouTube — backed by real web research.

## The team (5 agents)

| Agent | Job |
|---|---|
| **planner** | Plans the work, decides when the team is done (says `TERMINATE`). |
| **researcher** | Calls SerpAPI to gather facts about the topic. |
| **linkedin_writer** | Writes a professional LinkedIn post. |
| **instagram_writer** | Writes a short Instagram caption + hashtags. |
| **youtube_writer** | Writes a YouTube video title + 30-second script. |

Routing is handled by `SelectorGroupChat` — an LLM picks the next speaker each turn based on each agent's `description`.

## What you need in `.env`
```
OPENAI_API_KEY=sk-...
SERPAPI_API_KEY=...
```

## 1. Setup

In [ ]:
import os
import requests
from dotenv import load_dotenv
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing in .env"
assert os.getenv("SERPAPI_API_KEY"), "SERPAPI_API_KEY missing in .env"

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

## 2. SerpAPI tool

A plain Python function — AutoGen turns it into a tool the researcher can call. Keep the docstring sharp; that's what the LLM reads to decide when to use it.

In [ ]:
def search_web(query: str) -> str:
    """Search the web with SerpAPI and return the top 5 results as bullet points.
    Use this to gather current facts, statistics, or recent news about a topic.
    """
    r = requests.get(
        "https://serpapi.com/search",
        params={
            "q": query,
            "api_key": os.getenv("SERPAPI_API_KEY"),
            "num": 5,
            "engine": "google",
        },
        timeout=20,
    )
    r.raise_for_status()
    data = r.json()

    lines = []
    for item in data.get("organic_results", [])[:5]:
        title = item.get("title", "")
        snippet = item.get("snippet", "")
        link = item.get("link", "")
        lines.append(f"- {title}\n  {snippet}\n  ({link})")

    return "\n".join(lines) if lines else "No results found."

# Smoke test
print(search_web("latest news on AI agents")[:400])

## 3. Define the 5 agents

`description` is what the selector reads to choose the next speaker — keep them concrete and non-overlapping. `system_message` is the agent's actual instruction.

In [ ]:
planner = AssistantAgent(
    name="planner",
    model_client=model_client,
    description="Plans the work, delegates to specialists, and ends the run with TERMINATE when all three posts are ready.",
    system_message=(
        "You coordinate a content team. On the first turn, write a 3-step plan:\n"
        "1) researcher gathers facts, 2) the three writers each produce their post, 3) you wrap up.\n"
        "On later turns, only speak when all three posts (linkedin, instagram, youtube) are present in the chat.\n"
        "When they are, summarize the deliverables in a short list and end with the single word TERMINATE."
    ),
)

researcher = AssistantAgent(
    name="researcher",
    model_client=model_client,
    description="Gathers facts about the topic by calling the search_web tool.",
    tools=[search_web],
    system_message=(
        "Call search_web at least once on the topic. Then post a short brief: 5 bullet points of facts "
        "the writers can use. Do not write any posts yourself."
    ),
)

linkedin_writer = AssistantAgent(
    name="linkedin_writer",
    model_client=model_client,
    description="Writes a professional LinkedIn post for the topic, using the researcher's brief.",
    system_message=(
        "Write a LinkedIn post (120-200 words). Professional, first-person voice, one insight, "
        "end with a question to spark comments. Prefix the post with 'LINKEDIN POST:'."
    ),
)

instagram_writer = AssistantAgent(
    name="instagram_writer",
    model_client=model_client,
    description="Writes a short Instagram caption with hashtags, using the researcher's brief.",
    system_message=(
        "Write an Instagram caption (max 60 words), punchy and visual, plus 5-8 hashtags. "
        "Prefix with 'INSTAGRAM CAPTION:'."
    ),
)

youtube_writer = AssistantAgent(
    name="youtube_writer",
    model_client=model_client,
    description="Writes a YouTube video title and a short script, using the researcher's brief.",
    system_message=(
        "Produce a YouTube video TITLE and a 30-second SCRIPT (~75 words) in spoken style. "
        "Prefix with 'YOUTUBE:'."
    ),
)

## 4. Build the team

`SelectorGroupChat` reads each agent's `description` to pick who speaks next. Termination fires when the planner says `TERMINATE`, or after 12 messages as a safety net.

In [ ]:
team = SelectorGroupChat(
    participants=[planner, researcher, linkedin_writer, instagram_writer, youtube_writer],
    model_client=model_client,
    termination_condition=TextMentionTermination("TERMINATE") | MaxMessageTermination(12),
    allow_repeated_speaker=False,
)

## 5. Run on a topic

In [ ]:
topic = "How small businesses can use AI agents in 2026"

result = await Console(team.run_stream(task=f"Topic: {topic}"))

## 6. Pull out just the deliverables

The full chat is in `result.messages` — filter to the writers' posts.

In [ ]:
writer_names = {"linkedin_writer", "instagram_writer", "youtube_writer"}

for m in result.messages:
    if m.source in writer_names:
        print(f"=== {m.source} ===\n{m.content}\n")

In [ ]:
await model_client.close()